# Notebook 4: Query Complexity and Cost-Aware Routing
## SmartCache ML: Routing Baseline, Scikit-Learn Classifier & Trade-offs

### 1. Problem Overview
Language models follow a steep capability vs cost curve:
- **Small models** (e.g., LLaMA-3.2-1B, Claude 3 Haiku, GPT-4o-mini) are 10x–20x cheaper and much faster, handling factual, simple, or formatting tasks cleanly.
- **Large models** (e.g., LLaMA-3.2-3B, Claude 3.5 Sonnet, GPT-4o) excel at multi-step reasoning, mathematical derivations, and complex system architectures, but are expensive and slower.

Routing queries intelligently according to **query complexity** reduces total system cost while preserving high answer quality.

---
### Learning Objectives
By the end of this notebook, you will be able to:
- Implement a rule-based routing baseline using query length and reasoning keyword heuristics.
- Train a lightweight Scikit-learn TF-IDF + Logistic Regression complexity classifier.
- Prevent data leakage by evaluating on group-partitioned train/validation/test sets.
- Analyze classifier evaluation metrics: Precision, Recall, F1, and Confusion Matrix.
- Configure probability routing thresholds to balance cost savings against risk of quality degradation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

from smartcache.config import settings
from smartcache.router import RuleBasedRouter, ClassifierRouter

# Load group-aware dataset splits
train_df = pd.read_csv(settings.data_processed_path / "train_queries.csv")
val_df = pd.read_csv(settings.data_processed_path / "val_queries.csv")
test_df = pd.read_csv(settings.data_processed_path / "test_queries.csv")

print(f"Train samples: {len(train_df)}")
print(f"Val samples:   {len(val_df)}")
print(f"Test samples:  {len(test_df)}")

### 2. Rule-Based Routing Baseline
Let's evaluate our heuristic baseline on the test set.

In [ ]:
rule_router = RuleBasedRouter(length_threshold_words=12)

y_test_true = test_df['complexity_label'].tolist()
y_test_rule = []

for q in test_df['query_text']:
    dec = rule_router.route(q)
    # Map 'small' -> 'simple', 'large' -> 'complex'
    pred_label = "simple" if dec.selected_route == "small" else "complex"
    y_test_rule.append(pred_label)

print("=== RULE-BASED ROUTER CLASSIFICATION REPORT ===")
print(classification_report(y_test_true, y_test_rule, digits=3))

### 3. Training the Scikit-Learn TF-IDF + Logistic Regression Classifier
Now let's train our lightweight ML classifier pipeline on the training split and evaluate it on the unseen test split.

In [ ]:
ml_router = ClassifierRouter(probability_threshold=0.50)
ml_router.train(train_df['query_text'], train_df['complexity_label'])

# Evaluate on test set
y_test_ml = []
confidences = []

for q in test_df['query_text']:
    dec = ml_router.route(q)
    pred_label = "simple" if dec.selected_route == "small" else "complex"
    y_test_ml.append(pred_label)
    confidences.append(dec.confidence)

print("=== TF-IDF + LOGISTIC REGRESSION CLASSIFICATION REPORT ===")
print(classification_report(y_test_true, y_test_ml, digits=3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

cm_rule = confusion_matrix(y_test_true, y_test_rule, labels=["simple", "complex"])
cm_ml = confusion_matrix(y_test_true, y_test_ml, labels=["simple", "complex"])

ConfusionMatrixDisplay(cm_rule, display_labels=["simple", "complex"]).plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title("Rule-Based Baseline")

ConfusionMatrixDisplay(cm_ml, display_labels=["simple", "complex"]).plot(ax=axes[1], cmap='Greens', colorbar=False)
axes[1].set_title("ML Classifier Router")

plt.tight_layout()
plt.show()

### 4. Model Interpretability: Top Feature Coefficients
Let's inspect which n-grams drive the classifier toward 'simple' vs 'complex'.

In [ ]:
tfidf = ml_router.pipeline.named_steps['tfidf']
clf = ml_router.pipeline.named_steps['clf']

feature_names = np.array(tfidf.get_feature_names_out())
coefs = clf.coef_[0]

# If 'complex' is class 1 (or 0)
classes = list(clf.classes_)
complex_idx = classes.index("complex")
if complex_idx == 1:
    top_complex = feature_names[np.argsort(coefs)[-10:]]
    top_simple = feature_names[np.argsort(coefs)[:10]]
else:
    top_complex = feature_names[np.argsort(coefs)[:10]]
    top_simple = feature_names[np.argsort(coefs)[-10:]]

print("Top indicators of COMPLEX queries:", list(top_complex))
print("Top indicators of SIMPLE queries: ", list(top_simple))

### 5. Interview Concepts Checklist
1. **Model Cascading vs Ensembling**: In cascading/routing, only ONE model is queried per request based on pre-filtering, drastically cutting inference cost. In ensembling, all models execute concurrently.
2. **Threshold Tuning for Risk Asymmetry**: If a complex query is misclassified as simple, answer quality degrades. Therefore, setting a conservative routing threshold (e.g. routing to large model whenever $P(\text{complex}) \ge 0.35$) provides safety against hallucinations.
3. **Cold-start & Inference Latency of the Router**: The TF-IDF + LogisticRegression pipeline executes in < 0.5 ms on CPU, adding virtually zero overhead to the query pipeline.